# 02b MyNeta data prep

Cleans the MyNeta candidate data (assets, liabilities, age, education, criminal cases) for Bihar, Maharashtra, Uttar Pradesh and Tamil Nadu: general elections 2014, 2019, 2024 plus the by-elections in between.

- **In:** `candidates_processed.csv` (after OCR). `candidates_all_years.csv` (raw scrape) is only used to see which values were images.
- **Out (`clean/`):** `myneta_candidates_clean_2026-10-20.csv` (one row per candidate per election), data dictionary, cleaning log.

Each step prints counts before and after and adds a line to the cleaning log (same format as the OpenCity log).

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

COLLECTION_DATE = "2026-10-20"

# look in the project folders first, then next to the notebook
def find_file(name):
    for folder in [Path("../interim"), Path("../raw/lok_sabha_candidate_dfs"), Path(".")]:
        if (folder / name).exists():
            return folder / name
    raise FileNotFoundError(name)

clean_dir = Path("../clean") if Path("../clean").is_dir() else Path("clean")
clean_dir.mkdir(exist_ok=True)

df = pd.read_csv(find_file("candidates_processed.csv"))
raw = pd.read_csv(find_file("candidates_all_years.csv"))

# same rows in the same order, ocr only filled in the blanks
assert len(df) == len(raw) and (df["candidate"] == raw["candidate"]).all()

cleaning_log = []

def log_step(year, step, before, after, note=""):
    cleaning_log.append({
        "year": year, "step": step, "count_before": before,
        "count_after": after, "rows_affected": abs(before - after), "note": note
    })

In [2]:
# the data before any change
print(df.shape)
print(df.dtypes.to_string())
df.head()

(10395, 15)
year                  int64
state                object
const                object
sno                   int64
candidate            object
is_winner              bool
party                object
criminal_cases        int64
education            object
age                   int64
total_assets         object
liabilities          object
is_byelection          bool
bye_election_date    object
election_date        object


,year,state,const,sno,candidate,is_winner,party,criminal_cases,education,age,total_assets,liabilities,is_byelection,bye_election_date,election_date
0,2024,BIHAR,ARARIA,6,Pradeep Kumar Singh,True,BJP,4,10th Pass,58,"Rs 1,60,59,806\n~1 Crore+.","Rs 16,79,230\n~16 Lacs+.",False,NaN,07-05-2024
1,2024,BIHAR,ARARIA,1,Akhilesh Kumar,False,IND,0,Doctorate,47,"Rs 2,47,11,288 ~ 2 Crore+","Rs 3,05,374 ~ 3 Lacs+",False,NaN,07-05-2024
2,2024,BIHAR,ARARIA,2,Md Ghousul Azam,False,BSP,0,10th Pass,38,"Rs 1,60,50,000 ~ 1 Crore+","Rs 14,00,000 ~ 14 Lacs+",False,NaN,07-05-2024
3,2024,BIHAR,ARARIA,3,Md. Ismail,False,Bharatiya Momin Front,0,Graduate Professional,56,"Rs 7,95,000\n~7 Lacs+.",Rs 0,False,NaN,07-05-2024
4,2024,BIHAR,ARARIA,4,Md. Mobinul Haque,False,IND,0,Post Graduate,51,"Rs 34,05,000 ~ 34 Lacs+",Rs 0 ~,False,NaN,07-05-2024


### Step 1: extra spaces and line breaks

OCR leaves line breaks (`"Rs 1,60,59,806\n~1 Crore+."`) and the web page leaves non-breaking spaces. Any run of whitespace becomes one space, ends trimmed.

In [3]:
def tidy(s):
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

for col in ["state", "const", "candidate", "party", "education", "total_assets", "liabilities"]:
    before = (df[col] != tidy(df[col])).sum()
    df[col] = tidy(df[col])
    after = (df[col] != tidy(df[col])).sum()
    log_step("all", f"{col}: extra spaces and line breaks", int(before), int(after), "cells with a line break, nbsp or double space")
    print(col, "messy cells before:", before, "after:", after)

state messy cells before: 0 after: 0
const messy cells before: 0 after: 0
candidate messy cells before: 0 after: 0
party messy cells before: 0 after: 0
education messy cells before: 0 after: 0
total_assets messy cells before: 10308 after: 0
liabilities messy cells before: 8578 after: 0


### Step 2: party spellings

Some parties are spelt two ways (`CPI(ML)L` / `CPI(ML)(L)`). Names that match once case and symbols are ignored get the most common spelling.

In [4]:
key = df["party"].str.lower().str.replace(r"[^a-z]", "", regex=True)

# show the groups that have more than one spelling
groups = df.groupby(key)["party"].unique()
print(groups[groups.map(len) > 1].to_string())

most_common = df.groupby(key)["party"].agg(lambda s: s.value_counts().index[0])
before = df["party"].nunique()
df["party"] = key.map(most_common)
after = df["party"].nunique()
log_step("all", "party: same name, different spelling", before, after, "distinct party names")
print("\ndistinct party names before:", before, "after:", after)

party
allpensionersparty               [All Pensioner’s Party, All Pensioners Party]
ambedkaritepartyofindia      [Ambedkarite Party of India, Ambedkarite Party...
ambedkarsamajparty                [Ambedkar Samaj Party, Ambedkar samaj Party]
cpimll                                                  [CPI(ML)(L), CPI(ML)L]
cpimlredstar                              [CPI(ML) Red Star, CPI(ML) Red star]
minoritiesdemocraticparty    [Minorities Democratic Party, Minorities democ...
peoplesunionparty                  [Peoples Union Party, People’s Union Party]
rashtrawadipartyofindia      [Rashtrawadi Party of India,, Rashtrawadi Part...
rashtriyajanadhikarparty     [Rashtriya Janadhikar Party, Rashtriya Jan Adh...
vanchitsamajinsaafparty      [Vanchitsamaj Insaaf Party, VANCHITSAMAJ INSAA...
welfarepartyofindia           [Welfare Party Of India, Welfare Party of India]

distinct party names before: 771 after: 760


### Step 3: duplicates

- **Key:** `year, state, const, is_byelection, sno` = one candidate in one election. `is_byelection` is needed because a by-election and the general election of the same seat and year both number from 1.
- **Same name twice in one seat:** kept, since age, party and assets differ (different people). Flagged in `namesake_in_seat`.
- **Across sources:** the OpenCity copy of a candidate is joined in the merge notebook, not removed here.

In [5]:
key_cols = ["year", "state", "const", "is_byelection", "sno"]

exact = df.duplicated().sum()
key_dups = df.duplicated(key_cols).sum()
log_step("all", "exact duplicate rows", int(exact), 0, "rows")
log_step("all", "duplicate keys (year, seat, by-election, sno)", int(key_dups), 0, "rows; one row per candidate per election")
print("exact duplicate rows:", exact, "| duplicate keys:", key_dups)

# without is_byelection the by-election rows clash with the general election rows
print("rows clashing if is_byelection is left out:", df.duplicated(["year", "state", "const", "sno"], keep=False).sum())

df["namesake_in_seat"] = df.duplicated(["year", "state", "const", "is_byelection", "candidate"], keep=False)
print("namesake rows:", df["namesake_in_seat"].sum())
df.loc[df["namesake_in_seat"], ["year", "const", "candidate", "party", "age", "total_assets"]].head(10)

exact duplicate rows: 0 | duplicate keys: 0
rows clashing if is_byelection is left out: 268
namesake rows: 26


,year,const,candidate,party,age,total_assets
157,2024,KARAKAT,Raja Ram Singh,CPI(ML)L,66,"Rs 2,59,19,942 ~ 2 Crore+"
166,2024,KARAKAT,Raja Ram Singh,IND,54,"Rs 26,17,500 ~ 26 Lacs+."
277,2024,NALANDA,Munna Kumar,Samajwadi Lok Parishad,30,"Rs 1,25,000 ~1 Lacs+."
278,2024,NALANDA,Munna Kumar,Sankhyanupati Bhagidari Party,39,"Rs 5,55,000 ~ 5 Lacs+"
1963,2024,KARUR,Jothimani. S,INC,48,"Rs 1,53,52,373 ~ 1 Crore+"
1973,2024,KARUR,Jothimani. S,IND,37,"Rs 51,33,450 ~ 51 Lacs+"
2146,2024,PERAMBALUR,Arun Nehru,DMK,40,"Rs 81,61,97,502 ~ 81 Crore+"
2148,2024,PERAMBALUR,Arun Nehru,IND,27,"Rs 98,80,000 ~98 Lacs+."
2169,2024,POLLACHI,Eswarasamy K,DMK,47,"Rs 34,52,76,957 ~ 34 Crores."
2171,2024,POLLACHI,Eswarasamy K,IND,59,"Rs 1,41,10,000 ~ 1 Crore+"


### Step 4: assets and liabilities to rupees

MyNeta prints `Rs 1,60,59,806 ~ 1 Crore+`. The parser keeps only the digits after `Rs` (so OCR spaces like `Rs 8,01 ,25,802` don't matter), then checks the number against the rounded value after `~`.

Status per value: `ok`, `nil` (= 0), `no_approx` (no rounded value printed), `mismatch`, `unreadable`. The last two must not occur.

In [ ]:
UNITS = {"crore": 1e7, "lac": 1e5, "lakh": 1e5, "thou": 1e3, "hund": 1e2}

def parse_money(text):
    if text.lower() == "nil":
        return 0, "nil"
    m = re.match(r"Rs\.?\s*(\d[\d,\s]*?)\s*(?:~\s*(.*))?$", text)
    if not m:
        return np.nan, "unreadable"
    value = int(re.sub(r"\D", "", m.group(1)))
    approx = re.match(r"([\d.]+)\s*([a-z]+)", (m.group(2) or "").lower())
    if not approx:
        return value, "no_approx"
    unit = next((u for k, u in UNITS.items() if approx.group(2).startswith(k)), None)
    ok = unit is not None and value // unit == int(float(approx.group(1)))
    return value, "ok" if ok else "mismatch"

# strings seen in the data
assert parse_money("Rs 1,60,59,806 ~1 Crore+.") == (16059806, "ok")
assert parse_money("Rs 8,01 ,25,802 ~ 8 Crores.") == (80125802, "ok")
assert parse_money("Rs 0 ~") == (0, "no_approx")
assert parse_money("Nil") == (0, "nil")

In [7]:
for col, new in [("total_assets", "assets"), ("liabilities", "liabilities")]:
    ocr_spaces = df[col].str.contains(r"\d\s+,|,\s+\d|\d\s+\d").sum()
    parsed = df[col].map(parse_money)
    df[new + "_rs"] = parsed.str[0]
    df[new + "_status"] = parsed.str[1]

    not_parsed = df[new + "_rs"].isna().sum()
    log_step("all", f"{col}: text to rupees", len(df), int(not_parsed), "cells stored as text")
    log_step("all", f"{col}: ocr spaces inside the number", int(ocr_spaces), 0, "cells; the old parser read these wrongly")
    print(col, "| spaces inside the number:", ocr_spaces, "| status:", df[new + "_status"].value_counts().to_dict())

# every value is checked against the rounded value or has a known reason
assert not df[["assets_status", "liabilities_status"]].isin(["mismatch", "unreadable"]).any().any()
df[["assets_rs", "liabilities_rs"]] = df[["assets_rs", "liabilities_rs"]].astype("int64")

# which values came from an image (every third row on the page)
df["asset_source"] = np.where(raw["asset_link"].notna(), "image (ocr)", "text")
print(pd.crosstab(raw["sno"] % 3, df["asset_source"]))

total_assets | spaces inside the number: 16 | status: {'ok': 10304, 'nil': 86, 'no_approx': 5}
liabilities | spaces inside the number: 3 | status: {'no_approx': 5772, 'ok': 4623}
asset_source  image (ocr)  text
sno                            
0                    3260     0
1                       0  3664
2                       0  3471


### Step 5: impossible values and dates

- **Age:** must be at least 25 for the Lok Sabha. 0 (unknown) and the two rows with 21 and 24 become missing.
- **Education:** `Not Given` becomes missing.
- **Criminal cases:** checked to be 0 or more.
- **Dates:** `dd-mm-yyyy` text to real dates.

In [8]:
age_before = df["age"].copy()

bad_age = df["age"] < 25
print("ages below 25:", df.loc[bad_age, "age"].value_counts().to_dict())
df["age"] = df["age"].where(~bad_age).astype("Int64")
log_step("all", "age: 0 or below 25 set to missing", int(bad_age.sum()), int((df["age"] < 25).sum()), "rows with an impossible age")

not_given = df["education"].eq("Not Given")
df["education"] = df["education"].mask(not_given)
log_step("all", "education: 'Not Given' set to missing", int(not_given.sum()), int(df["education"].eq("Not Given").sum()), "rows with 'Not Given'")
print("education 'Not Given':", not_given.sum())

assert (df["criminal_cases"] >= 0).all()

for col in ["election_date", "bye_election_date"]:
    df[col] = pd.to_datetime(df[col], format="%d-%m-%Y")
print(df[["election_date", "bye_election_date"]].describe().loc[["count", "min", "max"]])

ages below 25: {0: 39, 24: 1, 21: 1}
education 'Not Given': 81
             election_date    bye_election_date
count                10228                  167
min    2014-04-10 00:00:00  2014-09-13 00:00:00
max    2024-06-01 00:00:00  2024-11-20 00:00:00


### Step 6: missing values

Missing cells per column and year. The `asset image (before ocr)` row counts asset values that were images on MyNeta before OCR filled them in.

In [9]:
missing = df.isna().groupby(df["year"]).sum().T
missing.loc["asset image (before ocr)"] = raw["total_assets"].isna().groupby(raw["year"]).sum()
missing = missing[missing.sum(axis=1) > 0]
missing["rows"] = missing.sum(axis=1)
missing.loc["all rows"] = list(df["year"].value_counts().sort_index()) + [len(df)]
missing

year,2014,2019,2024,rows
education,63,10,8,81
age,29,12,0,41
bye_election_date,3611,3232,3385,10228
election_date,99,50,18,167
asset image (before ocr),1170,1021,1069,3260
all rows,3710,3282,3403,10395


In [10]:
# dates: empty exactly where they do not apply
print(pd.crosstab(df["is_byelection"], df["election_date"].isna()))
print(pd.crosstab(df["is_byelection"], df["bye_election_date"].isna()))

election_date  False  True 
is_byelection              
False          10228      0
True               0    167
bye_election_date  False  True 
is_byelection                  
False                  0  10228
True                 167      0


In [11]:
# is the missing pattern linked to year, state or winning? (chi-square test of independence)
def missing_test(flag, name):
    print(f"\n{name}: {flag.sum()} rows")
    for col in ["year", "state", "is_winner"]:
        chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(df[col], flag))
        share = (flag.groupby(df[col]).mean() * 100).round(1).to_dict()
        print(f"  {col}: chi-square = {chi2:.1f}, p = {p:.3g}, % missing {share}")
    # do the missing rows have different assets? (mann-whitney, medians in lakh)
    _, p = stats.mannwhitneyu(df.loc[flag, "assets_rs"], df.loc[~flag, "assets_rs"])
    print(f"  assets: median {df.loc[flag, 'assets_rs'].median() / 1e5:.1f} vs {df.loc[~flag, 'assets_rs'].median() / 1e5:.1f} lakh, p = {p:.3g}")

missing_test(raw["total_assets"].isna(), "asset shown as image")
missing_test(df["age"].isna(), "age missing")
missing_test(df["education"].isna(), "education missing")


asset shown as image: 3260 rows
  year: chi-square = 0.2, p = 0.926, % missing {2014: 31.5, 2019: 31.1, 2024: 31.4}
  state: chi-square = 0.8, p = 0.841, % missing {'BIHAR': 31.0, 'MAHARASHTRA': 31.5, 'TAMIL NADU': 32.0, 'UTTAR PRADESH': 30.9}
  is_winner: chi-square = 0.9, p = 0.346, % missing {False: 31.2, True: 33.1}
  assets: median 29.6 vs 28.8 lakh, p = 0.453

age missing: 41 rows
  year: chi-square = 27.7, p = 9.63e-07, % missing {2014: 0.8, 2019: 0.4, 2024: 0.0}
  state: chi-square = 38.0, p = 2.87e-08, % missing {'BIHAR': 0.3, 'MAHARASHTRA': 0.1, 'TAMIL NADU': 0.0, 'UTTAR PRADESH': 0.9}
  is_winner: chi-square = 1.7, p = 0.191, % missing {False: 0.4, True: 0.0}
  assets: median 20.1 vs 29.1 lakh, p = 0.0703

education missing: 81 rows
  year: chi-square = 63.1, p = 1.98e-14, % missing {2014: 1.7, 2019: 0.3, 2024: 0.2}
  state: chi-square = 16.6, p = 0.000868, % missing {'BIHAR': 0.3, 'MAHARASHTRA': 0.5, 'TAMIL NADU': 0.9, 'UTTAR PRADESH': 1.2}
  is_winner: chi-square = 4.3, p

**Why values are missing**
- **Dates:** `election_date` is empty for exactly the 167 by-election rows and `bye_election_date` for the 10,228 general rows.
- **Asset images (3,260 rows, 31%):** MyNeta shows every third row of a page as an image. No link with year (p = 0.93), state (p = 0.84), winning (p = 0.35) or assets (p = 0.45), so **MCAR**.
- **Age (41 rows, 0.4%):** only 2014 and 2019, mostly UP; linked to year and state (p < 0.001), no winners. **MAR**.
- **Education (81 rows, 0.8%):** mostly 2014, poorer candidates (median 4 vs 30 lakh), no winners. **MAR**, possibly **MNAR** (blank on purpose), which can't be tested.

**What we do**
- Asset images: read by OCR, so these are real values, not imputed. All 3,260 pass the Step 4 check.
- Age and education: left missing. Under 1% of rows, no winners, and the asset question doesn't use them. Imputing would invent facts about real people.
- Dates: left empty, they don't apply.

### Step 7: outliers

Assets range from Rs 9 to over Rs 1,000 crore, so the checks use a log scale:
- **Assets:** 1.5 x IQR on `log10(assets)`; zero assets counted separately.
- **Criminal cases:** most are 0, so IQR doesn't work. Modified z-score (median/MAD) > 3.5 on `log10(cases)` for candidates with at least one case.
- **Age:** 1.5 x IQR.

A flag only means "unusual"; each one is checked below.

In [12]:
def iqr_flags(s):
    q1, q3 = s.quantile([0.25, 0.75])
    low, high = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    return np.where(s < low, "low", np.where(s > high, "high", "none"))

# assets
positive = df["assets_rs"] > 0
df["assets_outlier"] = "none"
df.loc[positive, "assets_outlier"] = iqr_flags(np.log10(df.loc[positive, "assets_rs"]))
print("candidates declaring 0 assets:", (~positive).sum())
print("asset flags:", df["assets_outlier"].value_counts().to_dict())
print("status of flagged values:", df.loc[df["assets_outlier"] != "none", "assets_status"].value_counts().to_dict())
print(df.groupby("assets_outlier")["assets_rs"].agg(["min", "max"]))

cols = ["year", "const", "candidate", "party", "is_winner", "total_assets", "assets_rs"]
print("\nhighest:")
print(df[df["assets_outlier"] == "high"].nlargest(8, "assets_rs")[cols].to_string(index=False))
print("\nlowest:")
print(df[df["assets_outlier"] == "low"].nsmallest(8, "assets_rs")[cols].to_string(index=False))

candidates declaring 0 assets: 86
asset flags: {'none': 10310, 'low': 67, 'high': 18}
status of flagged values: {'ok': 80, 'no_approx': 5}
                       min          max
assets_outlier                         
high            2046415655  11075833190
low                      9         3500
none                     0   2011705843

highest:
 year         const                 candidate  party  is_winner                     total_assets   assets_rs
 2019   PATALIPUTRA       Ramesh Kumar Sharma    IND      False Rs 11,07,58,33,190 ~ 1107 Crore+ 11075833190
 2014     JAHANABAD         Anil Kumar Sharma  JD(U)      False   Rs 8,49,88,03,720 ~ 849 Crore+  8498803720
 2024         ERODE               Ashok Kumar AIADMK      False  Rs 6,62,46,87,500 ~ 662 Crores.  6624687500
 2019 KANNIYAKUMARI           Vasanthakumar H    INC       True   Rs 4,17,49,30,444 ~ 417 Crore+  4174930444
 2024      KOLHAPUR Chhatrapati Shahu Shahaji    INC       True  Rs 3,42,86,68,000 ~ 342 Crores.  34286680

In [13]:
# criminal cases
cases = np.log10(df.loc[df["criminal_cases"] > 0, "criminal_cases"])
z = 0.6745 * (cases - cases.median()) / (cases - cases.median()).abs().median()
df["criminal_cases_outlier"] = False
df.loc[z.index, "criminal_cases_outlier"] = z.abs() > 3.5
print("candidates with at least one case:", len(cases), "| flagged:", df["criminal_cases_outlier"].sum())
print(df.nlargest(6, "criminal_cases")[["year", "const", "candidate", "party", "criminal_cases", "criminal_cases_outlier"]].to_string(index=False))

# age
age_flags = iqr_flags(df["age"].dropna().astype(float))
print("\nage flags:", pd.Series(age_flags).value_counts().to_dict())
print(df.nlargest(5, "age")[["year", "const", "candidate", "party", "age"]].to_string(index=False))

candidates with at least one case: 2202 | flagged: 2
 year         const                       candidate party  criminal_cases  criminal_cases_outlier
 2014 KANNIYAKUMARI                 Udayakumar S.p.   AAP             382                    True
 2014  THOOTHUKKUDI                   M.pushparayan   AAP             380                    True
 2019      VARANASI                     Ateek Ahmad   IND              59                   False
 2014       PHULPUR                      Atiq Ahmad   IND              53                   False
 2014     SHRAWASTI                      Atiq Ahmad    SP              42                   False
 2024        PURNIA Rajesh Ranjan Alias Pappu Yadav   IND              41                   False

age flags: {'none': 10336, 'high': 18}
 year            const                    candidate                       party  age
 2014          HAJIPUR               Ram Sundar Das                       JD(U)   93
 2019          MADURAI                     Alagar P

**Diagnosis**
- **Assets:** 18 high (above ~Rs 200 crore), 67 low (Rs 9 to Rs 3,500). 80 of 85 match MyNeta's rounded value, so not typos; the other 5 are tiny amounts with no rounded value. High ones are known rich candidates (e.g. H. Vasanthakumar, Kanniyakumari 2019).
- **Criminal cases:** 382 (S.P. Udayakumar) and 380 (M. Pushparayan), both leaders of the Kudankulam protests where hundreds of cases were filed. Real.
- **Age:** up to 93 (Ram Sundar Das, Hajipur 2014). Real.

**Decision:** nothing changed or removed. Kept as flags in `assets_outlier` and `criminal_cases_outlier`; the dashboard uses medians and a log scale.

In [14]:
log_step("all", "outliers flagged (assets, IQR on log10)", int((df["assets_outlier"] != "none").sum()), 0, "rows flagged; all real values, none changed")
log_step("all", "outliers flagged (criminal cases, MAD on log10)", int(df["criminal_cases_outlier"].sum()), 0, "rows flagged; real counts, none changed")
log_step("all", "outliers flagged (age, IQR)", int((age_flags != "none").sum()), 0, "rows flagged; real ages, none changed")

### Step 8: save, dictionary, reload check

- **Folders:** `raw/` (scrape, never edited), `interim/` (OCR output), `clean/` (this table, dictionary, log).
- **Names:** source + date, e.g. `myneta_candidates_clean_2026-10-20.csv`.
- **Format:** CSV (about 10,400 rows, opens in Excel). CSV doesn't keep column types, so the file is read back with a `dtype` map and compared.

In [15]:
dictionary = pd.DataFrame([
    ("year", "int", "year", "MyNeta", "2014, 2019, 2024", "General election the row belongs to (by-elections carry the year of the general election before them)"),
    ("state", "str", "name", "MyNeta", "BIHAR, MAHARASHTRA, TAMIL NADU, UTTAR PRADESH", "State of the constituency, upper case"),
    ("const", "str", "name", "MyNeta", "non-empty", "Constituency name as on MyNeta"),
    ("sno", "int", "number", "MyNeta", ">= 1", "Row number on the MyNeta constituency page"),
    ("candidate", "str", "name", "MyNeta", "non-empty", "Candidate name as on MyNeta"),
    ("party", "str", "name", "MyNeta; spellings unified (Step 2)", "non-empty, IND = independent", "Party of the candidate"),
    ("is_winner", "bool", "flag", "MyNeta", "True / False", "Candidate won this election"),
    ("is_byelection", "bool", "flag", "MyNeta", "True / False", "Row is from a by-election"),
    ("election_date", "date", "date", "MyNeta", "2014 to 2024; empty for by-elections", "Polling date of the general election"),
    ("bye_election_date", "date", "date", "MyNeta", "2014 to 2024; empty for general elections", "Polling date of the by-election"),
    ("age", "Int64", "years", "MyNeta affidavit", "25 to 93, missing for 41 rows", "Age declared in the affidavit; 0 and under 25 set to missing"),
    ("education", "str", "category", "MyNeta affidavit", "Illiterate ... Doctorate, Others, missing", "Highest education declared; 'Not Given' set to missing"),
    ("criminal_cases", "int", "count", "MyNeta affidavit", ">= 0", "Number of criminal cases declared"),
    ("total_assets", "str", "text", "MyNeta (text or OCR of image)", "'Rs ... ~ ...' or 'Nil'", "Total assets as printed, spaces tidied"),
    ("assets_rs", "int", "rupees", "derived from total_assets", ">= 0", "Total assets in rupees (Nil = 0)"),
    ("assets_status", "str", "category", "derived", "ok, nil, no_approx", "Check of assets_rs against the rounded value MyNeta prints"),
    ("asset_source", "str", "category", "derived", "text, image (ocr)", "Whether assets and liabilities came as text or were read from an image"),
    ("liabilities", "str", "text", "MyNeta (text or OCR of image)", "'Rs ... ~ ...'", "Total liabilities as printed, spaces tidied"),
    ("liabilities_rs", "int", "rupees", "derived from liabilities", ">= 0", "Total liabilities in rupees"),
    ("liabilities_status", "str", "category", "derived", "ok, no_approx", "Check of liabilities_rs against the rounded value"),
    ("namesake_in_seat", "bool", "flag", "derived", "True / False", "Another candidate in the same seat and election has the same name"),
    ("assets_outlier", "str", "category", "derived (IQR on log10 assets)", "none, low, high", "Asset outlier flag; all diagnosed as real"),
    ("criminal_cases_outlier", "bool", "flag", "derived (MAD on log10 cases)", "True / False", "Criminal case outlier flag; diagnosed as real"),
], columns=["column", "dtype", "unit", "source", "allowed_range", "description"])

clean = df[dictionary["column"].tolist()]
assert list(clean.columns) == dictionary["column"].tolist()
dictionary

,column,dtype,unit,source,allowed_range,description
0,year,int,year,MyNeta,"2014, 2019, 2024",General election the row belongs to (by-electi...
1,state,str,name,MyNeta,"BIHAR, MAHARASHTRA, TAMIL NADU, UTTAR PRADESH","State of the constituency, upper case"
2,const,str,name,MyNeta,non-empty,Constituency name as on MyNeta
3,sno,int,number,MyNeta,>= 1,Row number on the MyNeta constituency page
4,candidate,str,name,MyNeta,non-empty,Candidate name as on MyNeta
5,party,str,name,MyNeta; spellings unified (Step 2),"non-empty, IND = independent",Party of the candidate
6,is_winner,bool,flag,MyNeta,True / False,Candidate won this election
7,is_byelection,bool,flag,MyNeta,True / False,Row is from a by-election
8,election_date,date,date,MyNeta,2014 to 2024; empty for by-elections,Polling date of the general election
9,bye_election_date,date,date,MyNeta,2014 to 2024; empty for general elections,Polling date of the by-election


In [16]:
# save, read back, and compare types and values
clean_path = clean_dir / f"myneta_candidates_clean_{COLLECTION_DATE}.csv"
clean.to_csv(clean_path, index=False)
dictionary.to_csv(clean_dir / f"myneta_data_dictionary_{COLLECTION_DATE}.csv", index=False)

dtype_map = {
    "year": "int64", "state": "string", "const": "string", "sno": "int64", "candidate": "string",
    "party": "string", "is_winner": "bool", "is_byelection": "bool", "age": "Int64", "education": "string",
    "criminal_cases": "int64", "total_assets": "string", "assets_rs": "int64", "assets_status": "string",
    "asset_source": "string", "liabilities": "string", "liabilities_rs": "int64", "liabilities_status": "string",
    "namesake_in_seat": "bool", "assets_outlier": "string", "criminal_cases_outlier": "bool",
}
back = pd.read_csv(clean_path, dtype=dtype_map, parse_dates=["election_date", "bye_election_date"])

expected = clean.astype(dtype_map)
print("shape equal:", back.shape == clean.shape)
print("all dtypes as expected after reading back:", (back.dtypes == expected.dtypes).all())
print("values equal:", back.astype(str).equals(expected.astype(str)))

# without the dtype map age comes back as float and the dates as text
plain = pd.read_csv(clean_path)
print("\nwithout a dtype map:", {c: str(plain[c].dtype) for c in ["age", "election_date"]})

shape equal: True
all dtypes as expected after reading back: True
values equal: True

without a dtype map: {'age': 'float64', 'election_date': 'object'}


In [17]:
clean_log = pd.DataFrame(cleaning_log)
clean_log.to_csv(clean_dir / f"myneta_cleaning_log_{COLLECTION_DATE}.csv", index=False)
clean_log

,year,step,count_before,count_after,rows_affected,note
0,all,state: extra spaces and line breaks,0,0,0,"cells with a line break, nbsp or double space"
1,all,const: extra spaces and line breaks,0,0,0,"cells with a line break, nbsp or double space"
2,all,candidate: extra spaces and line breaks,0,0,0,"cells with a line break, nbsp or double space"
3,all,party: extra spaces and line breaks,0,0,0,"cells with a line break, nbsp or double space"
4,all,education: extra spaces and line breaks,0,0,0,"cells with a line break, nbsp or double space"
5,all,total_assets: extra spaces and line breaks,10308,0,10308,"cells with a line break, nbsp or double space"
6,all,liabilities: extra spaces and line breaks,8578,0,8578,"cells with a line break, nbsp or double space"
7,all,"party: same name, different spelling",771,760,11,distinct party names
8,all,exact duplicate rows,0,0,0,rows
9,all,"duplicate keys (year, seat, by-election, sno)",0,0,0,rows; one row per candidate per election
